**This notebook is an exercise in the [Intermediate Machine Learning](https://www.kaggle.com/learn/intermediate-machine-learning) course.  You can reference the tutorial at [this link](https://www.kaggle.com/alexisbcook/missing-values).**

---


Now it's your turn to test your new knowledge of **missing values** handling. You'll probably find it makes a big difference.

# Setup

The questions will give you feedback on your work. Run the following cell to set up the feedback system.

In [1]:
# Set up code checking
import os
if not os.path.exists("../input/train.csv"):
    os.symlink("../input/home-data-for-ml-course/train.csv", "../input/train.csv")  
    os.symlink("../input/home-data-for-ml-course/test.csv", "../input/test.csv") 
from learntools.core import binder
binder.bind(globals())
from learntools.ml_intermediate.ex2 import *
print("Setup Complete")

Setup Complete


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Read the data
X_full = pd.read_csv('../input/train.csv', index_col='Id')
X_test_full = pd.read_csv('../input/test.csv', index_col='Id')

# Remove rows with missing target, separate target from predictors
X_full.dropna(axis=0, subset=['SalePrice'], inplace=True)
y = X_full.SalePrice
X_full.drop(['SalePrice'], axis=1, inplace=True)

# To keep things simple, we'll use only numerical predictors
X = X_full.select_dtypes(exclude=['object'])
X_test = X_test_full.select_dtypes(exclude=['object'])

# Break off validation set from training data
X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2,
                                                      random_state=0)

In this exercise, you will work with data from the [Housing Prices Competition for Kaggle Learn Users](https://www.kaggle.com/c/home-data-for-ml-course). 

![Ames Housing dataset image](https://storage.googleapis.com/kaggle-media/learn/images/lTJVG4e.png)

Run the next code cell without changes to load the training and validation sets in `X_train`, `X_valid`, `y_train`, and `y_valid`.  The test set is loaded in `X_test`.

Use the next code cell to print the first five rows of the data.

In [3]:
X_train.head()

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pan

,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,...,GarageArea,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold
Id,,,,,,,,,,,,,,,,,,,,,
619,20,90.0,11694,9,5,2007,2007,452.0,48,0,...,774,0,108,0,0,260,0,0,7,2007
871,20,60.0,6600,5,5,1962,1962,0.0,0,0,...,308,0,0,0,0,0,0,0,8,2009
93,30,80.0,13360,5,7,1921,2006,0.0,713,0,...,432,0,0,44,0,0,0,0,8,2009
818,20,NaN,13265,8,5,2002,2002,148.0,1218,0,...,857,150,59,0,0,0,0,0,7,2008
303,20,118.0,13704,7,5,2001,2002,150.0,0,0,...,843,468,81,0,0,0,0,0,1,2006


You can already see a few missing values in the first several rows.  In the next step, you'll obtain a more comprehensive understanding of the missing values in the dataset.

# Step 1: Preliminary investigation

Run the code cell below without changes.

In [4]:
# Shape of training data (num_rows, num_columns)
print(X_train.shape)

# Number of missing values in each column of training data
missing_val_count_by_column = (X_train.isnull().sum())
print(missing_val_count_by_column[missing_val_count_by_column > 0])

(1168, 36)
LotFrontage    212
MasVnrArea       6
GarageYrBlt     58
dtype: int64


### Part A

Use the above output to answer the questions below.

In [5]:
# Fill in the line below: How many rows are in the training data?
num_rows = 1168

# Fill in the line below: How many columns in the training data
# have missing values?
num_cols_with_missing = 3

# Fill in the line below: How many missing entries are contained in 
# all of the training data?
tot_missing = 276

# Check your answers
step_1.a.check()

<IPython.core.display.Javascript object>

<span style="color:#33cc33">Correct</span>

In [6]:
# Lines below will give you a hint or solution code
#step_1.a.hint()
#step_1.a.solution()

### Part B
Considering your answers above, what do you think is likely the best approach to dealing with the missing values?

In [7]:
# Check your answer (Run this code cell to receive credit!)
step_1.b.check()

<IPython.core.display.Javascript object>

<span style="color:#33cc33">Correct:</span> 

Since there are relatively few missing entries in the data (the column with the greatest percentage of missing values is missing less than 20% of its entries), we can expect that dropping columns is unlikely to yield good results.  This is because we'd be throwing away a lot of valuable data, and so imputation will likely perform better.

In [8]:
#step_1.b.hint()

To compare different approaches to dealing with missing values, you'll use the same `score_dataset()` function from the tutorial.  This function reports the [mean absolute error](https://en.wikipedia.org/wiki/Mean_absolute_error) (MAE) from a random forest model.

In [9]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Function for comparing different approaches
def score_dataset(X_train, X_valid, y_train, y_valid):
    model = RandomForestRegressor(n_estimators=100, random_state=0)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    return mean_absolute_error(y_valid, preds)

# Step 2: Drop columns with missing values

In this step, you'll preprocess the data in `X_train` and `X_valid` to remove columns with missing values.  Set the preprocessed DataFrames to `reduced_X_train` and `reduced_X_valid`, respectively.  

In [10]:
# Fill in the line below: get names of columns with missing values
cols_with_missing = [col for col in X_train.columns
                     if X_train[col].isnull().any()] # Your code here

# Fill in the lines below: drop columns in training and validation data
reduced_X_train = X_train.drop(cols_with_missing, axis=1)
reduced_X_valid = X_valid.drop(cols_with_missing, axis=1)

step_2.check()

<IPython.core.display.Javascript object>

<span style="color:#33cc33">Correct</span>

In [11]:
# Lines below will give you a hint or solution code
# step_2.hint()
# step_2.solution()

Run the next code cell without changes to obtain the MAE for this approach.

In [12]:
print("MAE (Drop columns with missing values):")
print(score_dataset(reduced_X_train, reduced_X_valid, y_train, y_valid))

MAE (Drop columns with missing values):
17837.82570776256


# Step 3: Imputation

### Part A

Use the next code cell to impute missing values with the mean value along each column.  Set the preprocessed DataFrames to `imputed_X_train` and `imputed_X_valid`.  Make sure that the column names match those in `X_train` and `X_valid`.

In [13]:
from sklearn.impute import SimpleImputer
import pandas as pd

# Fill in the lines below: imputation
simp_imputer = SimpleImputer() # Your code here
imputed_X_train = pd.DataFrame(simp_imputer.fit_transform(X_train))
imputed_X_valid = pd.DataFrame(simp_imputer.transform(X_valid))

# Fill in the lines below: imputation removed column names; put them back
imputed_X_train.columns = X_train.columns
imputed_X_valid.columns = X_valid.columns

# Check your answers
step_3.a.check()

<IPython.core.display.Javascript object>

<span style="color:#33cc33">Correct</span>

In [14]:
# Lines below will give you a hint or solution code
#step_3.a.hint()
#step_3.a.solution()

Run the next code cell without changes to obtain the MAE for this approach.

In [15]:
print("MAE (Imputation):")
print(score_dataset(imputed_X_train, imputed_X_valid, y_train, y_valid))

MAE (Imputation):
18062.894611872147


### Part B

Compare the MAE from each approach.  Does anything surprise you about the results?  Why do you think one approach performed better than the other?

In [16]:
# Check your answer (Run this code cell to receive credit!)
step_3.b.check()

<IPython.core.display.Javascript object>

<span style="color:#33cc33">Correct:</span> 

Given that thre are so few missing values in the dataset, we'd expect imputation to perform better than dropping columns entirely.  However, we see that dropping columns performs slightly better!  While this can probably partially be attributed to noise in the dataset, another potential explanation is that the imputation method is not a great match to this dataset.  That is, maybe instead of filling in the mean value, it makes more sense to set every missing value to a value of 0, to fill in the most frequently encountered value, or to use some other method.  For instance, consider the `GarageYrBlt` column (which indicates the year that the garage was built).  It's likely that in some cases, a missing value could indicate a house that does not have a garage.  Does it make more sense to fill in the median value along each column in this case?  Or could we get better results by filling in the minimum value along each column?  It's not quite clear what's best in this case, but perhaps we can rule out some options immediately - for instance, setting missing values in this column to 0 is likely to yield horrible results!

In [17]:
#step_3.b.hint()

# Step 4: Generate test predictions

In this final step, you'll use any approach of your choosing to deal with missing values.  Once you've preprocessed the training and validation features, you'll train and evaluate a random forest model.  Then, you'll preprocess the test data before generating predictions that can be submitted to the competition!

### Part A

Use the next code cell to preprocess the training and validation data.  Set the preprocessed DataFrames to `final_X_train` and `final_X_valid`.  **You can use any approach of your choosing here!**  in order for this step to be marked as correct, you need only ensure:
- the preprocessed DataFrames have the same number of columns,
- the preprocessed DataFrames have no missing values, 
- `final_X_train` and `y_train` have the same number of rows, and
- `final_X_valid` and `y_valid` have the same number of rows.

In [18]:
 # preprocess the data in `X_train` and `X_valid` to remove columns with missing values.  Set the preprocessed DataFrames to `reduced_X_train` and `reduced_X_valid`, respectively.  
missValCols = [col for col in X_train.columns if X_train[col].isnull().any()]

# Fill in the lines below: drop columns in training and validation data
reduced_X_train = X_train.drop(missValCols, axis=1)
reduced_X_valid = X_valid.drop(missValCols, axis=1)

print('Col count in X_train: ', reduced_X_train.shape[1])
print('Col count in X_Valid: ', reduced_X_valid.shape[1])
print('Row count in reduced_X_train: ', reduced_X_train.shape[0])
print('Row count in reduced_X_valid: ', reduced_X_valid.shape[0])
print('Row count in y_train: ', y_train.shape[-1])
print('Row count in y_valid: ', y_valid.shape[-1])
# Preprocessed training and validation features
final_X_train = reduced_X_train
final_X_valid = reduced_X_valid

# Check your answers
step_4.a.check()

Col count in X_train:  33
Col count in X_Valid:  33
Row count in reduced_X_train:  1168
Row count in reduced_X_valid:  292
Row count in y_train:  1168
Row count in y_valid:  292


<IPython.core.display.Javascript object>

<span style="color:#33cc33">Correct</span>

In [19]:
 # With SimpleImputer 

# Fill in the lines below: impute columns in training and validation data
imp_X_train = pd.DataFrame(simp_imputer.fit_transform(X_train))
imp_X_valid = pd.DataFrame(simp_imputer.transform(X_valid))

print('Col count in X_train: ', imp_X_train.shape[1])
print('Col count in X_Valid: ', imp_X_valid.shape[1])
print('Row count in imp_X_train: ', imp_X_train.shape[0])
print('Row count in reduced_X_valid: ', imp_X_valid.shape[0])
print('Row count in y_train: ', y_train.shape[-1])
print('Row count in y_valid: ', y_valid.shape[-1])
# Preprocessed training and validation features
final_X_train_imp = imp_X_train
final_X_valid_imp = imp_X_valid

# Check your answers
step_4.a.check()

Col count in X_train:  36
Col count in X_Valid:  36
Row count in imp_X_train:  1168
Row count in reduced_X_valid:  292
Row count in y_train:  1168
Row count in y_valid:  292


<IPython.core.display.Javascript object>

<span style="color:#33cc33">Correct</span>

In [20]:
# Lines below will give you a hint or solution code
#step_4.a.hint()
#step_4.a.solution()

Run the next code cell to train and evaluate a random forest model.  (*Note that we don't use the `score_dataset()` function above, because we will soon use the trained model to generate test predictions!*)

In [21]:
# Define and fit model
model = RandomForestRegressor(n_estimators=100, random_state=0)
model.fit(final_X_train, y_train)     #dropped cols
model_imp = RandomForestRegressor(n_estimators=100, random_state=0)
model_imp.fit(final_X_train_imp, y_train) #imputed cols

# Get validation predictions and MAE
preds_valid = model.predict(final_X_valid)
preds_valid_imp = model_imp.predict(final_X_valid_imp)
print("MAE (Dropped Data):")
print(mean_absolute_error(y_valid, preds_valid))
print("MAE (Imputed Data):")
print(mean_absolute_error(y_valid, preds_valid_imp))

MAE (Dropped Data):
17837.82570776256
MAE (Imputed Data):
18062.894611872147


In [22]:
# Make copy to avoid changing original data (when imputing)
X_train_plus = X_train.copy()
X_valid_plus = X_valid.copy()

# Make new columns indicating what will be imputed
for col in cols_with_missing:
    X_train_plus[col + '_was_missing'] = X_train_plus[col].isnull()
    X_valid_plus[col + '_was_missing'] = X_valid_plus[col].isnull()

# Imputation
my_imputer = SimpleImputer()
imputed_X_train_plus = pd.DataFrame(my_imputer.fit_transform(X_train_plus))
imputed_X_valid_plus = pd.DataFrame(my_imputer.transform(X_valid_plus))

# Imputation removed column names; put them back
imputed_X_train_plus.columns = X_train_plus.columns
imputed_X_valid_plus.columns = X_valid_plus.columns

print("MAE from Approach 3 (An Extension to Imputation):")
print(score_dataset(imputed_X_train_plus, imputed_X_valid_plus, y_train, y_valid))

MAE from Approach 3 (An Extension to Imputation):
18148.417180365297


### Part B

Use the next code cell to preprocess your test data.  Make sure that you use a method that agrees with how you preprocessed the training and validation data, and set the preprocessed test features to `final_X_test`.

Then, use the preprocessed test features and the trained model to generate test predictions in `preds_test`.

In order for this step to be marked correct, you need only ensure:
- the preprocessed test DataFrame has no missing values, and
- `final_X_test` has the same number of rows as `X_test`.

In [23]:
# Since MAE score less for dropping columns, WE PROCEED WITH DROPPING COLUMNS
# Remove rows with missing target, separate target from predictors ## reminder of X, y, X_test,
# X_full.dropna(axis=0, subset=['SalePrice'], inplace=True)
# y = X_full.SalePrice
# X_full.drop(['SalePrice'], axis=1, inplace=True)

# # To keep things simple, we'll use only numerical predictors
# X = X_full.select_dtypes(exclude=['object'])
# X_test = X_test_full.select_dtypes(exclude=['object'])

# Fill in the lines below: drop columns in training and validation data
miss_val_cols_test = [col for col in X_test.columns if X_test[col].isnull().any()]
print('Missing columns in Test Data',miss_val_cols_test)
# Number of missing values in each column of training data
miss_val_cols_count = (X_test.isnull().sum())
print(miss_val_cols_count[miss_val_cols_count > 0])

# drop missing columnar data
reduced_X_test = X_test.drop(miss_val_cols_test, axis=1)

# final_X_test has the same number of rows as X_test.
print('Shape X_test: ', X_test.shape)
print('Shape reduced_X_test: ', reduced_X_test.shape)

# # Fill in the line below: preprocess test data
final_X_test = reduced_X_test

# when I go to predict my model throws 'missing columns during fit' error
# preds_test = model.predict(final_X_test)
# Break off validation set from training data
# X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=0)

# Align test_df to match the exact columns the model saw during fit
X_test_aligned = X_test.reindex(columns=model.feature_names_in_, fill_value=0).fillna(0)

# Now predict safely
preds_test = model.predict(X_test_aligned)

# # Check your answers
step_4.b.check()

Missing columns in Test Data ['LotFrontage', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'GarageYrBlt', 'GarageCars', 'GarageArea']
LotFrontage     227
MasVnrArea       15
BsmtFinSF1        1
BsmtFinSF2        1
BsmtUnfSF         1
TotalBsmtSF       1
BsmtFullBath      2
BsmtHalfBath      2
GarageYrBlt      78
GarageCars        1
GarageArea        1
dtype: int64
Shape X_test:  (1459, 36)
Shape reduced_X_test:  (1459, 25)


<IPython.core.display.Javascript object>

<span style="color:#33cc33">Correct</span>

In [24]:
# Save test predictions to file
output_test = pd.DataFrame({'Id': X_test.index,
                       'SalePrice': preds_test})
output_test.head()

,Id,SalePrice
0,1461,126915.50
1,1462,157737.00
2,1463,181556.40
3,1464,184109.00
4,1465,197827.95


In [25]:
# Since MAE score less for dropping columns, WE PROCEED WITH DROPPING COLUMNS

# Fill in the lines below: drop columns in training and validation data
miss_val_cols_test = [col for col in X_test.columns if X_test[col].isnull().any()]
print('Missing columns in Test Data',miss_val_cols_test)
# Number of missing values in each column of training data
miss_val_cols_count = (X_test.isnull().sum())
print(miss_val_cols_count[miss_val_cols_count > 0])

# drop missing columnar data
reduced_X_test = X_test.drop(miss_val_cols_test, axis=1)

# final_X_test has the same number of rows as X_test.
print('Shape X_test: ', X_test.shape)
print('Shape reduced_X_test: ', reduced_X_test.shape)

# # Fill in the line below: preprocess test data
final_X_test = reduced_X_test

# when I go to predict my model throws missing columns during fit error
# preds_test = model.predict(final_X_test)

# SO WE PROCEED TO TRANSFORM USING IMPUTATION -- ANOTHER WAY TO PREDICT

reduced_X_tst = pd.DataFrame(simp_imputer.transform(X_test))

# Get test predictions
preds_test = model_imp.predict(reduced_X_tst)

# # Check your answers
step_4.b.check()

Missing columns in Test Data ['LotFrontage', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'GarageYrBlt', 'GarageCars', 'GarageArea']
LotFrontage     227
MasVnrArea       15
BsmtFinSF1        1
BsmtFinSF2        1
BsmtUnfSF         1
TotalBsmtSF       1
BsmtFullBath      2
BsmtHalfBath      2
GarageYrBlt      78
GarageCars        1
GarageArea        1
dtype: int64
Shape X_test:  (1459, 36)
Shape reduced_X_test:  (1459, 25)


<IPython.core.display.Javascript object>

<span style="color:#33cc33">Correct</span>

In [26]:
# Lines below will give you a hint or solution code
# step_4.b.hint()
# step_4.b.solution()

Run the next code cell without changes to save your results to a CSV file that can be submitted directly to the competition.

In [27]:
# Save test predictions to file
output = pd.DataFrame({'Id': X_test.index,
                       'SalePrice': preds_test})
output.to_csv('submission.csv', index=False)

In [28]:
output.head()

,Id,SalePrice
0,1461,125245.50
1,1462,155237.00
2,1463,180755.22
3,1464,184071.50
4,1465,197144.40


<!-- 
## All In All ##

1. I tried to drop columns and got good MAE
2. I tried to impute columns and got ok MAE
3. When I tried (1) with test data - it WILL NOT have y (SalePrice) - that is what I have to predict
4. I faced error during model predictions with more columns in test data being null.
5. Refitting the model does not work as source data is a bit different; SO I REINDEX - to avoid imputation
6. The resultant predictions are generated
7. I retry with imputation and get better results - this is submitted


 -->

# Submit your results

Once you have successfully completed Step 4, you're ready to submit your results to the leaderboard!  (_You also learned how to do this in the previous exercise.  If you need a reminder of how to do this, please use the instructions below._)  

First, you'll need to join the competition if you haven't already.  So open a new window by clicking on [this link](https://www.kaggle.com/c/home-data-for-ml-course).  Then click on the **Join Competition** button.

![join competition image](https://storage.googleapis.com/kaggle-media/learn/images/wLmFtH3.png)

Next, follow the instructions below:
1. Begin by clicking on the **Save Version** button in the top right corner of the window.  This will generate a pop-up window.  
2. Ensure that the **Save and Run All** option is selected, and then click on the **Save** button.
3. This generates a window in the bottom left corner of the notebook.  After it has finished running, click on the number to the right of the **Save Version** button.  This pulls up a list of versions on the right of the screen.  Click on the ellipsis **(...)** to the right of the most recent version, and select **Open in Viewer**.  This brings you into view mode of the same page. You will need to scroll down to get back to these instructions.
4. Click on the **Data** tab near the top of the screen.  Then, click on the file you would like to submit, and click on the **Submit** button to submit your results to the leaderboard.

You have now successfully submitted to the competition!

If you want to keep working to improve your performance, select the **Edit** button in the top right of the screen. Then you can change your code and repeat the process. There's a lot of room to improve, and you will climb up the leaderboard as you work.


# Keep going

Move on to learn what **[categorical variables](https://www.kaggle.com/alexisbcook/categorical-variables)** are, along with how to incorporate them into your machine learning models.  Categorical variables are very common in real-world data, but you'll get an error if you try to plug them into your models without processing them first!

---




*Have questions or comments? Visit the [course discussion forum](https://www.kaggle.com/learn/intermediate-machine-learning/discussion) to chat with other learners.*